# NOAA-18 / NOAA-19 SST gap filling: train the remaining sectors (Kaggle)

This is the same pipeline that trained **Goa, Tamil Nadu, South AP, North AP and West Bengal**
on the local GTX 1050 Ti, now for the other sectors:

| id | sector | id | sector | id | sector |
|---|---|---|---|---|---|
| 1 | Gujarat | 5 | Kerala | 12 | Lakshadweep |
| 2 | Maharashtra | 6 | South Tamil Nadu | 13 | North Andaman |
| 4 | Karnataka | 10 | Odisha | 14 | South Andaman |

**What it does:** one model per satellite and sector.
* **Train:** 2014-01-01 → 2016-06-30, with every 10th day held out for validation.
* **Test:** 2016-07-01 → 2016-12-31 (6 months, never trained on).
* **Fill:** any scene is filled from the **30 days of that satellite's passes before it**.

**You only need** the six zip files: `n18_2014-*.zip n18_2015-*.zip n18_2016-*.zip
n19_2014-*.zip n19_2015-*.zip n19_2016-*.zip` (~7.7 GB total).

**Runs twice?** Every step skips work that is already done (archives, finished models).
If the session dies, run all cells again and it continues.

**Splitting across accounts:** set `SATS = ["n18"]` in one account and `SATS = ["n19"]` in the
other, or split `SECTORS`. Then copy both `models/` folders together at the end.

## 0. Before you start
1. **Add Input → Upload → New Dataset**: upload the six zips; name it e.g. `noaa-sst`.
   Kaggle may unpack the zips into folders; either form works.
2. **Settings → Accelerator → GPU T4 x2** (NOAA-18 and NOAA-19 then train in parallel,
   one per GPU). **Settings → Internet → On** (for `pip install netCDF4`).
3. For an unattended run: **Save Version → Save & Run All (Commit)**. Everything in
   `/kaggle/working` becomes the notebook's output and can be downloaded.

Rough time for all 9 sectors × 2 satellites on T4 x2: about 20 min to build the archives,
then about 1 h of training (limit 12 h).

## 1. Settings

In [ ]:
SATS    = ["n18", "n19"]
SECTORS = ["1", "2", "4", "5", "6", "10", "12", "13", "14"]   # the sectors not trained yet
EPOCHS      = 25     # same as the first 5 sectors (keep equal so the sectors are comparable)
MAX_MINUTES = 30     # per model safety cap
BATCH_SIZE  = 16

## 2. Paths, GPU check, netCDF4

In [ ]:
import os, sys, glob, shutil
DATA_DIR  = "/kaggle/input"                     # searched recursively for the zips / .nc files
WORK_DIR  = "/kaggle/working/output"            # saved as the notebook output
CACHE_DIR = "/tmp/feature_cache"                # large scratch, not saved
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs("/kaggle/working/gapfill_run/gapfill", exist_ok=True)
%cd /kaggle/working/gapfill_run
!nvidia-smi --query-gpu=name,memory.total --format=csv
try:
    import netCDF4
except ImportError:
    !pip -q install netCDF4
n_zip = len(glob.glob(os.path.join(DATA_DIR, "**", "n1[89]_*.zip"), recursive=True))
n_nc  = len(glob.glob(os.path.join(DATA_DIR, "**", "*-noaa-1[89]-sst.nc"), recursive=True))
print(f"found {n_zip} zips and {n_nc} extracted .nc files under {DATA_DIR}")
assert n_zip or n_nc, "attach the dataset with the n18_/n19_ files (Add Input)"

## 3. Write the pipeline code
Exact copies of `gapfill/*.py` from the project; nothing to upload. The sector grids + land masks of all 14 sectors are embedded as base64 (31 KB).

In [ ]:
%%writefile gapfill/common.py
"""
Shared pieces of the NOAA-18 / NOAA-19 AVHRR SST gap-filling pipeline.

Everything that must be identical at training time and at inference time lives
here: sector grids, the regridding of a raw swath onto a sector grid, the input
features built from the previous 30 days, and the network itself. The Dash app
imports this module, so the app and the trainer cannot drift apart.
"""
import os
import re
import numpy as np
import pandas as pd

HERE      = os.path.dirname(os.path.abspath(__file__))
NOVA_DIR  = os.path.dirname(HERE)
# Raw data: the n18_<year>-*.zip / n19_<year>-*.zip files (or their extracted
# .nc files), searched recursively. Outputs go under WORK_DIR.
# Both can be overridden, e.g. on Colab/Kaggle.
DATA_DIR    = os.environ.get("GAPFILL_DATA_DIR", NOVA_DIR)
WORK_DIR    = os.environ.get("GAPFILL_WORK_DIR", NOVA_DIR)
ARCHIVE_DIR = os.path.join(WORK_DIR, "archives")      # <sat>/<sector>.nc
MODEL_DIR   = os.environ.get("GAPFILL_MODEL_DIR", os.path.join(WORK_DIR, "models"))  # <sat>/<sector>.pt
CACHE_DIR   = os.environ.get("GAPFILL_CACHE_DIR", os.path.join(WORK_DIR, "feature_cache"))
GRID_FILE   = os.path.join(HERE, "sector_grids.npz")  # lat/lon/sea mask of all 14 sectors

SATELLITES = {"n18": "NOAA-18", "n19": "NOAA-19"}

# All 14 sectors. Ids/names/boxes are the ones in app/gapgan/dashapp.py; the
# grids (~0.036 deg) and land masks come from app/gapgan/nc_data/3R_<id>_SST_DATA.nc,
# copied into sector_grids.npz so this folder is self-contained.
SECTORS = {
    "1":  {"name": "1_Gujarat",            "label": "Gujarat",
           "bounds": [[20.04315567, 67.88010406], [23.75766563, 72.97928619]]},
    "2":  {"name": "2_Maharashtra",        "label": "Maharashtra",
           "bounds": [[15.70783329, 70.50718689], [20.04315567, 73.71035767]]},
    "3":  {"name": "3_Goa",                "label": "Goa",
           "bounds": [[14.85914707, 71.58411407], [15.70611858, 75.03230286]]},
    "4":  {"name": "4_Karnataka",          "label": "Karnataka",
           "bounds": [[12.80841446, 71.58411407], [14.86688137, 75.03580475]]},
    "5":  {"name": "5_Kerala",             "label": "Kerala",
           "bounds": [[7.38235903, 74.23501587], [12.79460812, 77.18965912]]},
    "6":  {"name": "6_SouthTamilNadu",     "label": "South Tamil Nadu",
           "bounds": [[7.01648331, 77.18965912], [10.06777668, 81.33168793]]},
    "7":  {"name": "7_NorthTamilNadu",     "label": "Tamil Nadu",
           "bounds": [[10.08158302, 79.02596283], [13.51026344, 82.38100433]]},
    "8":  {"name": "8_SouthAndhraPradesh", "label": "South Andhra Pradesh",
           "bounds": [[13.51026344, 79.87512970], [16.53046989, 83.54584503]]},
    "9":  {"name": "9_NorthAndhraPradesh", "label": "North Andhra Pradesh",
           "bounds": [[16.53046989, 82.18675232], [19.11697006, 85.88069916]]},
    "10": {"name": "10_Odisha",            "label": "Odisha",
           "bounds": [[19.11697006, 84.77572632], [22.21802711, 89.00102234]]},
    "11": {"name": "11_WestBengal",        "label": "West Bengal",
           "bounds": [[20.71798897, 87.48213196], [22.21802711, 89.00102234]]},
    "12": {"name": "12_lakshadweep",       "label": "Lakshadweep",
           "bounds": [[8.06546974, 71.29454803], [12.32047653, 74.05455780]]},
    "13": {"name": "13_NorthAndaman",      "label": "North Andaman",
           "bounds": [[10.07467461, 91.52107239], [15.23840141, 94.50333405]]},
    "14": {"name": "14_SouthAndaman",      "label": "South Andaman",
           "bounds": [[6.34685087, 92.12857056], [10.07467461, 94.77947235]]},
}
FIRST_BATCH = ["3", "7", "8", "9", "11"]                  # trained on the local GPU
REMAINING   = [s for s in SECTORS if s not in FIRST_BATCH]  # 1 2 4 5 6 10 12 13 14

TRAIN_END   = pd.Timestamp(os.environ.get("GAPFILL_TRAIN_END", "2016-07-01"))
TEST_START  = TRAIN_END                    # train: before TRAIN_END, test: from it on
CONTEXT_DAYS = 30                          # history the model may look at
N_PREV       = 4                           # most recent passes fed individually
MIN_CELL_FRAC = 0.3                        # fine pixels needed for a valid coarse cell


def load_sector_grid(sector_id):
    """lat (descending), lon (ascending), sea mask (bool) of a sector."""
    with np.load(GRID_FILE) as g:
        return (g[f"{sector_id}_lat"].copy(), g[f"{sector_id}_lon"].copy(),
                g[f"{sector_id}_mask"].astype(bool))


def parse_time(filename):
    """
    Observation time from a file name. Accepts the NOAA naming used in the
    archives (20161015-120450Z-noaa-18-sst.nc) and the ISRO naming used by the
    existing app (3RIMG_24NOV2025_2045_...).
    """
    base = os.path.basename(filename)
    m = re.search(r"(\d{8})-(\d{6})Z", base)
    if m:
        return pd.to_datetime(m.group(1) + m.group(2), format="%Y%m%d%H%M%S")
    m = re.search(r"(\d{2}[A-Za-z]{3}\d{4})_(\d{4})", base)
    if m:
        return pd.to_datetime(m.group(1) + m.group(2), format="%d%b%Y%H%M")
    m = re.search(r"(\d{8})[T_-]?(\d{4})", base)
    if m:
        return pd.to_datetime(m.group(1) + m.group(2), format="%Y%m%d%H%M")
    raise ValueError(f"Cannot read the date/time from '{base}'. Expected a name like "
                     f"20161015-120450Z-noaa-18-sst.tif")


def parse_satellite(filename):
    m = re.search(r"noaa-?(18|19)", os.path.basename(filename), re.I)
    return f"n{m.group(1)}" if m else None


# ---------------------------------------------------------------------------
# Regridding: fine swath (~0.01 deg) -> sector grid (~0.036 deg), box average
# ---------------------------------------------------------------------------
class Binner:
    """Averages every fine pixel whose centre falls in a coarse cell."""

    def __init__(self, fine_lat, fine_lon, lat, lon):
        dlat, dlon = abs(lat[1] - lat[0]), abs(lon[1] - lon[0])
        # lat is descending on the sector grid
        ri = np.rint((lat[0] - fine_lat) / dlat).astype(int)
        ci = np.rint((fine_lon - lon[0]) / dlon).astype(int)
        rows = np.where((ri >= 0) & (ri < len(lat)))[0]
        cols = np.where((ci >= 0) & (ci < len(lon)))[0]
        if len(rows) == 0 or len(cols) == 0:
            raise ValueError("The file does not cover this sector.")
        self.r0, self.r1 = rows.min(), rows.max() + 1
        self.c0, self.c1 = cols.min(), cols.max() + 1
        R, C = np.meshgrid(ri[self.r0:self.r1], ci[self.c0:self.c1], indexing="ij")
        self.shape = (len(lat), len(lon))
        self.flat = (R * len(lon) + C).ravel()
        self.total = np.bincount(self.flat, minlength=lat.size * lon.size)

    def __call__(self, fine):
        """fine: 2-D window [r0:r1, c0:c1], NaN = missing. Returns coarse field."""
        v = fine.ravel()
        ok = np.isfinite(v)
        n = self.total.size
        s = np.bincount(self.flat[ok], weights=v[ok], minlength=n)
        c = np.bincount(self.flat[ok], minlength=n)
        out = np.full(n, np.nan, dtype="f4")
        good = (c > 0) & (c >= MIN_CELL_FRAC * np.maximum(self.total, 1))
        out[good] = s[good] / c[good]
        return out.reshape(self.shape)


# ---------------------------------------------------------------------------
# Input features for one target pass (training and inference share this)
# ---------------------------------------------------------------------------
def is_day(times, lon_center):
    """Local solar time between 06 and 18 h -> daytime pass."""
    t = pd.DatetimeIndex(times)
    hour = (t.hour + t.minute / 60.0 + lon_center / 15.0) % 24
    return np.asarray((hour >= 6) & (hour < 18))


def n_channels():
    return 2 + 3 * N_PREV + 6 + 6


def build_features(target_obs, target_time, hist_obs, hist_times, stats, lat, lon):
    """
    target_obs : (H, W) float, NaN = gap (the image to fill)
    target_time: Timestamp
    hist_obs   : (T, H, W) float, NaN = gap, passes strictly BEFORE target_time
                 within CONTEXT_DAYS (older ones are ignored)
    hist_times : (T,) datetime64
    stats      : dict(mean=, std=) of SST over the training period
    Returns (C, H, W) float32.
    """
    mu, sd = stats["mean"], stats["std"]
    H, W = target_obs.shape
    lon_c = float(np.mean(lon))
    hist_times = pd.DatetimeIndex(hist_times)
    tt = pd.Timestamp(target_time)
    keep = (hist_times < tt) & (hist_times >= tt - pd.Timedelta(days=CONTEXT_DAYS))
    hist_obs, hist_times = hist_obs[keep], hist_times[keep]

    def norm(a):
        m = np.isfinite(a)
        return np.where(m, (a - mu) / sd, 0.0).astype("f4"), m.astype("f4")

    feats = []
    x, m = norm(target_obs)
    feats += [x, m]

    # the N_PREV most recent passes that saw anything
    seen = np.isfinite(hist_obs).reshape(len(hist_obs), -1).any(1) if len(hist_obs) else []
    idx = [i for i in range(len(hist_obs) - 1, -1, -1) if seen[i]][:N_PREV]
    for k in range(N_PREV):
        if k < len(idx):
            i = idx[k]
            x, m = norm(hist_obs[i])
            lag = (tt - hist_times[i]).total_seconds() / 86400.0 / CONTEXT_DAYS
            feats += [x, m, np.full((H, W), lag, "f4") * m]
        else:
            feats += [np.zeros((H, W), "f4")] * 3

    # composites: 30 d all passes, 30 d same day/night phase, last 7 days
    day_t = is_day([tt], lon_c)[0]
    day_h = is_day(hist_times, lon_c) if len(hist_times) else np.zeros(0, bool)
    recent = (hist_times >= tt - pd.Timedelta(days=7)) if len(hist_times) else np.zeros(0, bool)
    for sel in (np.ones(len(hist_obs), bool), day_h == day_t, recent):
        sub = hist_obs[sel]
        if len(sub):
            ok = np.isfinite(sub)
            cnt = ok.sum(0)
            s = np.where(ok, sub, 0).sum(0)
            mean = np.where(cnt > 0, s / np.maximum(cnt, 1), np.nan)
            x, m = norm(mean)
            frac = (cnt / max(len(sub), 1)).astype("f4")
            feats += [x, np.minimum(frac * 4, 1).astype("f4") * m]
        else:
            feats += [np.zeros((H, W), "f4")] * 2

    # static / time planes
    LAT = np.broadcast_to(((lat - 15) / 10)[:, None], (H, W))
    LON = np.broadcast_to(((lon - 80) / 10)[None, :], (H, W))
    doy = 2 * np.pi * tt.dayofyear / 365.25
    hr = 2 * np.pi * (tt.hour + tt.minute / 60) / 24
    for v in (LAT, LON):
        feats.append(np.asarray(v, "f4"))
    for v in (np.sin(doy), np.cos(doy), np.sin(hr), np.cos(hr)):
        feats.append(np.full((H, W), v, "f4"))
    out = np.stack(feats).astype("f4")
    assert out.shape[0] == n_channels(), out.shape
    return out


# ---------------------------------------------------------------------------
# Network: DINCAE-style convolutional auto-encoder with skip connections.
# Outputs a mean and a log-variance per pixel (Gaussian NLL), like DINCAE, but
# without DINCAE-1's dense bottleneck (which is >99% of its parameters and does
# not fit a 4 GB GPU); DINCAE 2 dropped it for the same reason.
# ---------------------------------------------------------------------------
def make_model(in_ch=None, base=32):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    in_ch = in_ch or n_channels()

    def block(i, o):
        return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.LeakyReLU(0.1),
                             nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.LeakyReLU(0.1))

    class GapFillNet(nn.Module):
        def __init__(self):
            super().__init__()
            f = [base, base * 2, base * 4, base * 8]
            self.enc = nn.ModuleList([block(in_ch, f[0]), block(f[0], f[1]),
                                      block(f[1], f[2]), block(f[2], f[3])])
            self.mid = block(f[3], f[3])
            self.dec = nn.ModuleList([block(f[3] + f[3], f[3]), block(f[3] + f[2], f[2]),
                                      block(f[2] + f[1], f[1]), block(f[1] + f[0], f[0])])
            self.head = nn.Conv2d(f[0], 2, 1)

        def forward(self, x):
            H, W = x.shape[-2:]
            ph, pw = (-H) % 16, (-W) % 16
            x = F.pad(x, (0, pw, 0, ph), mode="replicate")
            skips = []
            for e in self.enc:
                x = e(x); skips.append(x); x = F.max_pool2d(x, 2)
            x = self.mid(x)
            for d, s in zip(self.dec, reversed(skips)):
                x = F.interpolate(x, size=s.shape[-2:], mode="bilinear", align_corners=False)
                x = d(torch.cat([x, s], 1))
            out = self.head(x)[..., :H, :W]
            mean = out[:, 0]
            logvar = out[:, 1].clamp(-10, 5)
            return mean, logvar

    return GapFillNet()

In [ ]:
%%writefile gapfill/prepare_data.py
"""
Build per-satellite, per-sector SST archives from the raw NOAA data.

    python gapfill/prepare_data.py                          # all 14 sectors, both satellites
    python gapfill/prepare_data.py --sectors 1,2,4 --sats n18

Finds the data anywhere under $GAPFILL_DATA_DIR (default: this folder), either
as the original zips (n18_2014-*.zip ...) or as extracted files
(.../20140124-232440Z-noaa-18-sst.nc) -- Kaggle unpacks uploaded zips. Each
swath is cropped to the sectors, box-averaged onto the sector grid and written
to $GAPFILL_WORK_DIR/archives/<sat>/<sector_name>.nc with dims (time, lat, lon).
Passes that see none of a sector's sea are dropped for that sector.
"""
import os
import re
import sys
import glob
import time
import zipfile
import argparse
from multiprocessing import Pool

import numpy as np
import pandas as pd
import xarray as xr

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common as C

FILE_RE = re.compile(r"(\d{8}-\d{6}Z)-noaa-(18|19)-sst\.nc$")


def find_sources(sat):
    """[(zip_path or None, [member names or file paths])], de-duplicated by pass."""
    num = sat[1:]
    seen, tasks = set(), []
    zips = [z for z in glob.glob(os.path.join(C.DATA_DIR, "**", "*.zip"), recursive=True)
            if os.path.basename(z).lower().startswith(sat)]
    for zp in sorted(zips):
        with zipfile.ZipFile(zp) as z:
            names = sorted(n for n in z.namelist()
                           if (m := FILE_RE.search(n)) and m.group(2) == num)
        names = [n for n in names if FILE_RE.search(n).group(1) not in seen]
        seen.update(FILE_RE.search(n).group(1) for n in names)
        for i in range(0, len(names), 200):
            tasks.append((zp, names[i:i + 200]))
    loose = sorted(f for f in glob.glob(os.path.join(C.DATA_DIR, "**", f"*-noaa-{num}-sst.nc"),
                                        recursive=True)
                   if FILE_RE.search(f).group(1) not in seen)
    for i in range(0, len(loose), 200):
        tasks.append((None, loose[i:i + 200]))
    n = sum(len(t[1]) for t in tasks)
    print(f"{sat}: {n} passes in {len(zips)} zip(s) + {len(loose)} loose file(s)", flush=True)
    return tasks


def work(args):
    (zpath, names), sectors = args
    import netCDF4
    grids = {sid: C.load_sector_grid(sid) for sid in sectors}
    binners, out = {}, {sid: ([], []) for sid in sectors}
    z = zipfile.ZipFile(zpath) if zpath else None
    try:
        for name in names:
            try:
                t = C.parse_time(name)
                raw = z.read(name) if z else open(name, "rb").read()
                with netCDF4.Dataset("mem", memory=raw) as d:
                    if not binners:
                        flat, flon = d["lat"][:].data, d["lon"][:].data
                        binners = {sid: C.Binner(flat, flon, g[0], g[1])
                                   for sid, g in grids.items()}
                    var = d["SST"]
                    var.set_auto_mask(False)
                    for sid, b in binners.items():
                        win = var[0, b.r0:b.r1, b.c0:b.c1].astype("f4")
                        win[(win < -50) | (win > 50)] = np.nan
                        g = b(win)
                        sea = grids[sid][2]
                        if np.isfinite(g[sea]).any():
                            g[~sea] = np.nan
                            out[sid][0].append(t)
                            out[sid][1].append(g)
            except Exception as e:
                print(f"  skip {name}: {e}", flush=True)
    finally:
        if z:
            z.close()
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sats", default=",".join(C.SATELLITES))
    ap.add_argument("--sectors", default=",".join(C.SECTORS))
    ap.add_argument("--workers", type=int, default=min(8, os.cpu_count() or 2))
    a = ap.parse_args()
    sectors = a.sectors.split(",")
    t0 = time.time()
    for sat in a.sats.split(","):
        tasks = find_sources(sat)
        if not tasks:
            print(f"  no {sat} data found under {C.DATA_DIR}")
            continue
        acc = {sid: ([], []) for sid in sectors}
        with Pool(a.workers) as pool:
            for k, out in enumerate(pool.imap_unordered(work, [(t, sectors) for t in tasks]), 1):
                for sid, (ts, gs) in out.items():
                    acc[sid][0].extend(ts)
                    acc[sid][1].extend(gs)
                if k % 5 == 0 or k == len(tasks):
                    print(f"  [{k}/{len(tasks)}] {time.time()-t0:.0f}s", flush=True)

        for sid in sectors:
            ts, gs = acc.pop(sid)
            if not ts:
                print(f"{sat} {C.SECTORS[sid]['name']}: no passes")
                continue
            lat, lon, sea = C.load_sector_grid(sid)
            order = np.argsort(np.array(ts, dtype="datetime64[ns]"))
            times = pd.DatetimeIndex(np.array(ts, dtype="datetime64[ns]")[order])
            sst = np.stack([gs[i] for i in order])
            del gs
            # a swath split across two files can give duplicate timestamps
            _, first = np.unique(times.values, return_index=True)
            times, sst = times[first], sst[first]
            ds = xr.Dataset(
                {"SST": (("time", "lat", "lon"), sst.astype("f4")),
                 "mask": (("lat", "lon"), sea.astype("i1"))},
                coords={"time": times, "lat": lat, "lon": lon},
                attrs={"satellite": C.SATELLITES[sat], "sector_id": sid,
                       "sector_name": C.SECTORS[sid]["name"],
                       "source": "NOAA AVHRR L2 SST swaths, box-averaged to the sector grid"})
            os.makedirs(os.path.join(C.ARCHIVE_DIR, sat), exist_ok=True)
            path = os.path.join(C.ARCHIVE_DIR, sat, f"{C.SECTORS[sid]['name']}.nc")
            ds.to_netcdf(path, encoding={"SST": {"zlib": True, "complevel": 4}})
            cov = np.isfinite(sst[:, sea]).mean(1)
            print(f"{sat} {C.SECTORS[sid]['name']:22s} passes={len(times):5d} "
                  f"{times[0]:%Y-%m-%d}..{times[-1]:%Y-%m-%d}  mean coverage={cov.mean():.2f}",
                  flush=True)
    print(f"done in {time.time()-t0:.0f}s")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile gapfill/train.py
"""
Train one gap-filling model per satellite and sector on the GPU.

    python gapfill/train.py                              # every sector, both satellites
    python gapfill/train.py --sat n18 --sector 1,2,4     # a list of sectors
    python gapfill/train.py --skip-done                  # resume: skip finished models

Split (by observation time):
    train  2014-01-01 .. 2016-06-30, except every 10th day
    val    every 10th day of that span   (model selection)
    test   2016-07-01 .. 2016-12-31      (reported only, never used to train)

Each target pass is given the 30 days of passes before it as context (see
common.build_features). Extra gaps are cut into the target using the real
cloud mask of another pass; the loss is the Gaussian negative log-likelihood
on the pixels that were hidden (plus a small weight on the visible ones).
"""
import os
import sys
import json
import time
import argparse

import numpy as np
import pandas as pd
import xarray as xr
import torch

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import common as C

CACHE_DIR = C.CACHE_DIR
MIN_COV_TRAIN = 0.05
MIN_COV_TEST = 0.10


def load_archive(sat, sid):
    path = os.path.join(C.ARCHIVE_DIR, sat, f"{C.SECTORS[sid]['name']}.nc")
    with xr.open_dataset(path) as ds:
        return (pd.DatetimeIndex(ds.time.values), ds.SST.values.astype("f4"),
                ds.lat.values, ds.lon.values, ds.mask.values == 1)


def context_features(times, sst, lat, lon, stats, targets, cache_path):
    """(N, C-2, H, W) float16 memmap of the context channels of every target."""
    N, (H, W) = len(targets), sst.shape[1:]
    shape = (N, C.n_channels() - 2, H, W)
    meta = cache_path + ".json"
    if os.path.exists(cache_path) and os.path.exists(meta):
        m = json.load(open(meta))
        if m.get("shape") == list(shape) and m.get("stats") == stats:
            return np.lib.format.open_memmap(cache_path, mode="r")
    mm = np.lib.format.open_memmap(cache_path, mode="w+", dtype="f2", shape=shape)
    t0 = time.time()
    for k, i in enumerate(targets):
        lo = times.searchsorted(times[i] - pd.Timedelta(days=C.CONTEXT_DAYS))
        f = C.build_features(sst[i], times[i], sst[lo:i], times[lo:i], stats, lat, lon)
        mm[k] = f[2:]
        if k % 500 == 0:
            print(f"    features {k}/{N}  {time.time()-t0:.0f}s", flush=True)
    mm.flush()
    json.dump({"shape": list(shape), "stats": stats}, open(meta, "w"))
    return np.lib.format.open_memmap(cache_path, mode="r")


def make_batch(idx, ctx, tgt, tgt_ok, donors_ok, sea, rng, fixed=None):
    """Returns inputs, target, hidden mask, visible mask (torch, on CPU)."""
    B = len(idx)
    H, W = tgt.shape[1:]
    x = np.empty((B, C.n_channels(), H, W), "f4")
    hid = np.zeros((B, H, W), bool)
    vis = np.zeros((B, H, W), bool)
    for b, i in enumerate(idx):
        ok = tgt_ok[i]
        for _ in range(10):
            j = fixed[i] if fixed is not None else rng.integers(len(donors_ok))
            v = ok & donors_ok[j]
            h = ok & ~donors_ok[j]
            if h.sum() >= 0.05 * ok.sum() and v.sum() >= 0.02 * sea.sum():
                break
            if fixed is not None:
                break
        vis[b], hid[b] = v, h
        x[b, 0] = np.where(v, tgt[i], 0)
        x[b, 1] = v
        x[b, 2:] = ctx[i]
    return (torch.from_numpy(x), torch.from_numpy(np.nan_to_num(tgt[idx])),
            torch.from_numpy(hid), torch.from_numpy(vis))


def nll(mean, logvar, y, w):
    l = 0.5 * (logvar + (y - mean) ** 2 * torch.exp(-logvar))
    return (l * w).sum() / w.sum().clamp(min=1)


def evaluate(model, sel, ctx, tgt, tgt_ok, donors_ok, sea, stats, fixed, dev, bs=16):
    """RMSE (deg C) on the hidden pixels, model vs. 30-day composite baseline."""
    model.eval()
    se_m = se_b = ae_m = bias = n = 0.0
    with torch.no_grad():
        for s in range(0, len(sel), bs):
            idx = sel[s:s + bs]
            x, y, h, v = make_batch(idx, ctx, tgt, tgt_ok, donors_ok, sea, None, fixed)
            mean, _ = model(x.to(dev))
            mean = mean.cpu().numpy() * stats["std"]
            y = y.numpy() * stats["std"]
            h = h.numpy()
            # baseline: 30-day same day/night composite, else 30-day all, else mean
            # channels 14/15 = 30-day mean/frac, 16/17 = same-phase mean/frac
            xn = x.numpy()
            base = np.where(xn[:, 17] > 0, xn[:, 16],
                            np.where(xn[:, 15] > 0, xn[:, 14], 0)) * stats["std"]
            d = (mean - y)[h]
            se_m += (d ** 2).sum(); ae_m += np.abs(d).sum(); bias += d.sum()
            se_b += ((base - y)[h] ** 2).sum(); n += h.sum()
    model.train()
    n = max(n, 1)
    return dict(rmse=float(np.sqrt(se_m / n)), mae=float(ae_m / n), bias=float(bias / n),
                rmse_30day_composite=float(np.sqrt(se_b / n)), n_pixels=int(n),
                n_passes=int(len(sel)))


def train_one(sat, sid, epochs, bs, dev, max_minutes, amp=False):
    name = C.SECTORS[sid]["name"]
    print(f"\n=== {sat} {name} ===", flush=True)
    times, sst, lat, lon, sea = load_archive(sat, sid)
    ok = np.isfinite(sst) & sea
    cov = ok.reshape(len(sst), -1).sum(1) / sea.sum()

    is_train_period = times < C.TRAIN_END
    day_no = (times.normalize() - pd.Timestamp("2014-01-01")).days
    is_val = is_train_period & (day_no % 10 == 0)
    tr = np.where(is_train_period & ~is_val & (cov >= MIN_COV_TRAIN))[0]
    va = np.where(is_val & (cov >= MIN_COV_TRAIN))[0]
    te = np.where((times >= C.TEST_START) & (cov >= MIN_COV_TEST))[0]
    print(f"passes: train={len(tr)} val={len(va)} test={len(te)}  grid={sst.shape[1:]}", flush=True)

    vals = sst[is_train_period][ok[is_train_period]]
    stats = {"mean": float(vals.mean()), "std": float(vals.std())}

    targets = np.concatenate([tr, va, te])
    os.makedirs(CACHE_DIR, exist_ok=True)
    ctx_all = context_features(times, sst, lat, lon, stats, targets,
                               os.path.join(CACHE_DIR, f"{sat}_{name}.npy"))
    pos = {t: k for k, t in enumerate(targets)}
    # everything indexed by position in `targets`
    tgt = np.where(ok[targets], (sst[targets] - stats["mean"]) / stats["std"], np.nan).astype("f4")
    tgt_ok = ok[targets]
    donors_ok = ok[tr[cov[tr] < 0.95]] | ~sea           # real cloud masks from training passes
    P_tr = np.array([pos[t] for t in tr], dtype=int)
    P_va = np.array([pos[t] for t in va], dtype=int)
    P_te = np.array([pos[t] for t in te], dtype=int)
    rng_fix = np.random.default_rng(0)
    fixed = rng_fix.integers(len(donors_ok), size=len(targets))

    ctx = ctx_all                      # disk-backed; the OS page cache keeps it warm

    torch.manual_seed(0)
    model = C.make_model().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    steps = epochs * int(np.ceil(len(P_tr) / bs))
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=1e-3, total_steps=steps, pct_start=0.1)
    rng = np.random.default_rng(1)
    # mixed precision on tensor-core GPUs (T4, V100, A100...): same model, ~2x faster
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    out_dir = os.path.join(C.MODEL_DIR, sat)
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, f"{name}.pt")
    best, hist, t0 = 1e9, [], time.time()
    for ep in range(epochs):
        perm = rng.permutation(P_tr)
        tot = 0.0
        for s in range(0, len(perm), bs):
            x, y, h, v = make_batch(perm[s:s + bs], ctx, tgt, tgt_ok, donors_ok, sea, rng)
            x, y, h, v = x.to(dev), y.to(dev), h.to(dev), v.to(dev)
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                mean, logvar = model(x)
            loss = nll(mean.float(), logvar.float(), y, h.float() + 0.2 * v.float())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(x)
        vm = evaluate(model, P_va, ctx, tgt, tgt_ok, donors_ok, sea, stats, fixed, dev)
        el = (time.time() - t0) / 60
        hist.append({"epoch": ep + 1, "train_nll": tot / len(perm), **{f"val_{k}": v for k, v in vm.items()}})
        flag = ""
        if vm["rmse"] < best:
            best = vm["rmse"]; flag = "  *saved"
            torch.save({"state_dict": model.state_dict(), "stats": stats, "sat": sat,
                        "sector": sid, "epoch": ep + 1, "n_channels": C.n_channels()}, out_path)
        print(f"  ep {ep+1:3d}/{epochs} nll={tot/len(perm):.3f} val_rmse={vm['rmse']:.3f} "
              f"(30d-composite {vm['rmse_30day_composite']:.3f})  {el:.1f} min{flag}", flush=True)
        if el > max_minutes:
            print(f"  time budget of {max_minutes} min reached", flush=True)
            break

    ck = torch.load(out_path, map_location=dev, weights_only=False)
    model.load_state_dict(ck["state_dict"])
    val = evaluate(model, P_va, ctx, tgt, tgt_ok, donors_ok, sea, stats, fixed, dev)
    test = evaluate(model, P_te, ctx, tgt, tgt_ok, donors_ok, sea, stats, fixed, dev)
    metrics = {"satellite": C.SATELLITES[sat], "sector": name, "best_epoch": ck["epoch"],
               "train_passes": int(len(tr)), "stats": stats,
               "train_period": f"{times[0]:%Y-%m-%d}..{C.TRAIN_END - pd.Timedelta(days=1):%Y-%m-%d}"
                               f" (every 10th day held out)",
               "test_period": f"{C.TEST_START:%Y-%m-%d}..{times[-1]:%Y-%m-%d}",
               "validation": val, "test": test, "history": hist,
               "minutes": round((time.time() - t0) / 60, 1)}
    json.dump(metrics, open(out_path.replace(".pt", "_metrics.json"), "w"), indent=1)
    print(f"  VAL  rmse={val['rmse']:.3f} C  (30-day composite {val['rmse_30day_composite']:.3f})")
    print(f"  TEST rmse={test['rmse']:.3f} C  (30-day composite {test['rmse_30day_composite']:.3f})"
          f"  bias={test['bias']:+.3f}  passes={test['n_passes']}", flush=True)
    del ctx
    return metrics


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sat", choices=list(C.SATELLITES), default=None)
    ap.add_argument("--sector", default=None, help="sector id, or a comma-separated list")
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--batch-size", type=int, default=16)
    ap.add_argument("--max-minutes", type=float, default=25)
    ap.add_argument("--skip-done", action="store_true",
                    help="skip models that already have a _metrics.json (resume after a disconnect)")
    ap.add_argument("--no-amp", action="store_true", help="disable mixed precision")
    a = ap.parse_args()
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    amp = (dev == "cuda" and not a.no_amp and torch.cuda.get_device_capability()[0] >= 7)
    print(f"device: {dev} {torch.cuda.get_device_name(0) if dev == 'cuda' else ''}"
          f"  mixed precision: {amp}", flush=True)
    sats = [a.sat] if a.sat else list(C.SATELLITES)
    secs = a.sector.split(",") if a.sector else list(C.SECTORS)
    summary = []
    for sat in sats:
        for sid in secs:
            name = C.SECTORS[sid]["name"]
            done = os.path.join(C.MODEL_DIR, sat, f"{name}_metrics.json")
            if a.skip_done and os.path.exists(done):
                print(f"\n=== {sat} {name}: already trained, skipping ===", flush=True)
                m = json.load(open(done))
            elif not os.path.exists(os.path.join(C.ARCHIVE_DIR, sat, f"{name}.nc")):
                print(f"\n=== {sat} {name}: no archive, run prepare_data.py first ===", flush=True)
                continue
            else:
                m = train_one(sat, sid, a.epochs, a.batch_size, dev, a.max_minutes, amp)
            summary.append({k: m[k] for k in ("satellite", "sector", "best_epoch", "minutes")} |
                           {"val_rmse": m["validation"]["rmse"], "test_rmse": m["test"]["rmse"],
                            "test_rmse_30day_composite": m["test"]["rmse_30day_composite"],
                            "test_bias": m["test"]["bias"], "test_passes": m["test"]["n_passes"]})
    print("\n" + pd.DataFrame(summary).to_string(index=False))


if __name__ == "__main__":
    main()

In [ ]:
import base64, numpy as np
GRIDS_B64 = '...'
open('gapfill/sector_grids.npz', 'wb').write(base64.b64decode(GRIDS_B64))
g = np.load('gapfill/sector_grids.npz')
print('sector grids:', sorted({k.split('_')[0] for k in g.files}, key=int))

## 4. (nothing to copy: Kaggle inputs are already on local disk)

## 5. Build the sector archives

In [ ]:
# Builds archives/<sat>/<sector>.nc (the sector time series) from the zips.
# Only missing archives are built, so a re-run after a disconnect is quick.
import subprocess
env = dict(os.environ, GAPFILL_DATA_DIR=DATA_DIR, GAPFILL_WORK_DIR=WORK_DIR,
           GAPFILL_CACHE_DIR=CACHE_DIR, PYTHONUNBUFFERED="1")
sys.path.insert(0, "gapfill")
import common as C
for sat in SATS:
    todo = [s for s in SECTORS if not os.path.exists(
        os.path.join(WORK_DIR, "archives", sat, C.SECTORS[s]["name"] + ".nc"))]
    if not todo:
        print(f"{sat}: all archives already built"); continue
    print(f"{sat}: building {todo}")
    r = subprocess.run([sys.executable, "gapfill/prepare_data.py", "--sats", sat,
                        "--sectors", ",".join(todo)], env=env)
    assert r.returncode == 0, "prepare_data failed - see the output above"

## 6. Train (GPU)

In [ ]:
# One process per satellite. With 2 GPUs (Kaggle T4 x2) they run in parallel, one per GPU;
# with 1 GPU they run one after the other. --skip-done makes a re-run continue where it stopped.
import subprocess, sys, time, torch
env = dict(os.environ, GAPFILL_DATA_DIR=DATA_DIR, GAPFILL_WORK_DIR=WORK_DIR,
           GAPFILL_CACHE_DIR=CACHE_DIR, PYTHONUNBUFFERED="1")
n_gpu = torch.cuda.device_count()
print("GPUs:", n_gpu, [torch.cuda.get_device_name(i) for i in range(n_gpu)])
args = lambda sat: [sys.executable, "gapfill/train.py", "--sat", sat, "--sector", ",".join(SECTORS),
                    "--epochs", str(EPOCHS), "--max-minutes", str(MAX_MINUTES),
                    "--batch-size", str(BATCH_SIZE), "--skip-done"]
os.makedirs(os.path.join(WORK_DIR, "logs"), exist_ok=True)

if n_gpu >= 2 and len(SATS) > 1:
    procs, logs = [], []
    for i, sat in enumerate(SATS):
        log = os.path.join(WORK_DIR, "logs", f"train_{sat}.log")
        procs.append(subprocess.Popen(args(sat), env=dict(env, CUDA_VISIBLE_DEVICES=str(i % n_gpu)),
                                      stdout=open(log, "w"), stderr=subprocess.STDOUT))
        logs.append(log)
    shown = [0] * len(logs)
    while any(p.poll() is None for p in procs):
        time.sleep(30)
        for k, log in enumerate(logs):
            lines = open(log, encoding="utf-8", errors="replace").read().splitlines()
            for line in lines[shown[k]:]:
                if "===" in line or "TEST" in line or "VAL" in line or "Error" in line:
                    print(f"[{SATS[k]}] {line}", flush=True)
            shown[k] = len(lines)
    for k, log in enumerate(logs):
        print(f"\n----- {SATS[k]} (exit {procs[k].returncode}) -----")
        print("\n".join(open(log, encoding="utf-8", errors="replace").read().splitlines()[-14:]))
else:
    for sat in SATS:
        log = os.path.join(WORK_DIR, "logs", f"train_{sat}.log")
        with open(log, "a") as fh:
            p = subprocess.Popen(args(sat), env=env, stdout=subprocess.PIPE,
                                 stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:
                fh.write(line)
                if "features" not in line:
                    print(line, end="", flush=True)
            p.wait()
        print(f"{sat}: exit code {p.returncode}")

## 7. Results
Error on pixels hidden with real cloud masks, test period Jul–Dec 2016. The *30-day composite* column is the simple baseline the model should beat.

In [ ]:
import glob, json, pandas as pd
rows = []
for f in sorted(glob.glob(os.path.join(WORK_DIR, "models", "*", "*_metrics.json"))):
    m = json.load(open(f))
    rows.append({"satellite": m["satellite"], "sector": m["sector"], "best_epoch": m["best_epoch"],
                 "val RMSE °C": round(m["validation"]["rmse"], 3),
                 "test RMSE °C": round(m["test"]["rmse"], 3),
                 "30-day composite °C": round(m["test"]["rmse_30day_composite"], 3),
                 "test bias °C": round(m["test"]["bias"], 3), "test passes": m["test"]["n_passes"],
                 "minutes": m["minutes"]})
df = pd.DataFrame(rows)
df.to_csv(os.path.join(WORK_DIR, "metrics_summary.csv"), index=False)
df

In [ ]:
# one zip with everything you need at home (models + archives + metrics) -> Output tab
out_zip = shutil.make_archive("/kaggle/working/noaa_gapfill_outputs", "zip", WORK_DIR)
print(out_zip, round(os.path.getsize(out_zip) / 1e6), "MB")

## 8. Use the new models on your own machine
Copy the downloaded `models/` and `archives/` folders into the project folder `nova_data/`,
next to the ones already there (`models/n18/…`, `models/n19/…`, `archives/n18/…`).
Then start `run_noaa_app.bat`.

The new sectors turn **green** on the map. Select one, choose
*Scene from the archive* → Date → Time, and press *Run Reconstruction*.

To take the screenshots, run this with the app running:
`..\.venv\Scripts\python take_screenshots_noaa.py`